# WCU readout playground (LM / N)

Set the filter, masks and integration below, then run the cells to get a ScopeSim readout of the
WCU and display it. Uses the same set-up as the SIM scripts (`modules.sim_common`).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u

# put main_scripts/ (for `modules`) on the path
ROOT = Path.cwd().resolve()
while not (ROOT / "main_scripts").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "main_scripts"))

from modules import sim_common
from modules.config_common import load_observing_config
from modules.paths import project_root

sim_common.setup_scopesim()
observing = load_observing_config(project_root())

## Settings

- `obs_mode`: `wcu_img_lm` or `wcu_img_n`
- `fp_mask`: `open` (flat), `pinhole_lm`, `pinhole_n`, `grid_lm`
- `pp_mask`: e.g. `PPS-CFO2`, `SPM-LM`, `CLS-LM`, `SPM-N`, `CLS-N`; `psf_kernel`: `None` = `<pp_mask>_WCU`
- `nd_filter`: `None`, `ND_OD1` ... `ND_OD4`
- `dit` (s), `ndit`; `bb_aperture` (0 = closed, 1 = open); `bb_temp_k`

In [ ]:
instrument = {
    "obs_mode": "wcu_img_lm",
    "obs_filter": "Br_alpha",
    "fp_mask": "pinhole_lm",
    "pp_mask": "PPS-CFO2",
    "psf_kernel": None,
    "nd_filter": None,
    "dit": 0.04,
    "ndit": 1,
    "intrapixel_capacitance": True,
}
bb_aperture = 1.0
bb_temp_k = 1000
subtract_dark = True  # also read out a dark (BB aperture closed) and subtract it

In [ ]:
metis = sim_common.build_optical_train(instrument)
metis["wcu_source"].set_temperature(bb_temp=bb_temp_k * u.K)

frame, header = sim_common.readout_frame(metis, bb_aperture, instrument)
dark = sim_common.readout_frame(metis, 0.0, instrument)[0] if subtract_dark else None

saturation = sim_common.saturation_of(observing, sim_common.detector_key(instrument["obs_mode"]))
print(f"raw: median {np.median(frame):.4g}, max {np.max(frame):.4g} ADU "
      f"= {np.max(frame) / saturation:.3f} of saturation ({saturation:.0f} ADU)")

In [ ]:
image = frame - dark if subtract_dark else frame
title = (f"{instrument['obs_filter']}, {instrument['fp_mask']}, {instrument['pp_mask']}, "
         f"ND {instrument['nd_filter']}, DIT {instrument['dit']} s"
         + (" (dark-subtracted)" if subtract_dark else ""))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
lo, hi = np.percentile(image, (1, 99.9))
im = axes[0].imshow(image, origin="lower", vmin=lo, vmax=hi)
axes[0].set_title(title, fontsize=9)
fig.colorbar(im, ax=axes[0], label="ADU")
# zoom (log scale) around the brightest pixel, e.g. a pinhole PSF
half = 40  # half-width of the zoom (pixels)
y0, x0 = np.unravel_index(np.argmax(image), image.shape)
ys, xs = slice(max(y0 - half, 0), y0 + half + 1), slice(max(x0 - half, 0), x0 + half + 1)
cut = image[ys, xs]
im = axes[1].imshow(cut - np.min(cut) + 1, origin="lower", norm="log",
                    extent=(xs.start - 0.5, xs.stop - 0.5, ys.start - 0.5, ys.stop - 0.5))
axes[1].set_title(f"zoom on max at (x, y) = ({x0}, {y0}), log scale")
fig.colorbar(im, ax=axes[1], label="ADU - min + 1")
plt.show()